In [16]:
import os
import math
import re
import json
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

In [4]:
df_filter = pd.read_csv('results/results-camera/ds_ids_uncapable.csv')

In [24]:
def cosine_similarity_phrases(phrase1: str, phrase2: str) -> float:
    vectorizer = TfidfVectorizer(stop_words=None)
    try:
        vectors = vectorizer.fit_transform([phrase1, phrase2])
        similarity = cosine_similarity(vectors[0], vectors[1])[0][0]
    except:
        similarity = 1
    return similarity

def find_keywords(w):
    matches = re.findall(r'"(\w+)"', w)
    result = ' '.join(f'"{match}"' for match in matches)
    return result
    
def merge_message(p_k, p_v, m_k, m_v):
    similarity = cosine_similarity_phrases(find_keywords(m_v), find_keywords(p_v))
    return similarity > 0.5

def analysis_union(row):
    pyright = row["result_static_pyright_unjson"]
    mypy = row["result_static_mypy_unjson"]
    grammar = row["result_static_grammar_unjson"]
    if len(pyright) > 0 or len(mypy) > 0 or len(grammar) > 0:
        return 0
    return 1

def analysis_intersect(row):
    pyright = row["result_static_pyright_unjson"]
    mypy = row["result_static_mypy_unjson"]
    grammar = row["result_static_grammar_unjson"]
    if len(pyright) > 0 and len(mypy) > 0 and len(grammar) > 0:
        return 0
    return 1
    
def merge_static_analysis_union(row):
    pyright = row["result_static_pyright_unjson"]
    mypy = row["result_static_mypy_unjson"]
    grammar = row["result_static_grammar_unjson"]
    dct_simple = {}
    if isinstance(pyright, dict) and isinstance(mypy, dict):
        dct = {}
        for p_k, p_v in pyright.items():
            for m_k, m_v, in mypy.items():
                is_message = merge_message(p_k, p_v["message"], m_k, m_v["message"])
                if is_message:
                    dct[m_v["message"]] = m_k
                else:
                    dct[p_v["message"]] = p_k
        for a_k, a_v in dct.items():
            dct_simple[a_v] = a_k
    elif isinstance(pyright, dict) and not isinstance(mypy, dict):
        for p_k, p_v in pyright.items():
            dct_simple[p_k] = p_v["message"]
    elif not isinstance(pyright, dict) and isinstance(mypy, dict):
        for p_k, p_v in mypy.items():
            dct_simple[p_k] = p_v["message"]
    # Add grammar
    if isinstance(grammar, dict):
        for k, v in grammar.items():
            dct_simple[k] = v["message"]
    return dct_simple

def merge_static_analysis_intersection(row):
    pyright = row["result_static_pyright_unjson"]
    mypy = row["result_static_mypy_unjson"]
    grammar = row["result_static_grammar_unjson"]
    dct_simple = {}
    if isinstance(pyright, dict) and isinstance(mypy, dict):
        dct = {}
        for p_k, p_v in pyright.items():
            for m_k, m_v, in mypy.items():
                is_message = merge_message(p_k, p_v["message"], m_k, m_v["message"])
                if is_message:
                    dct[m_v["message"]] = m_k
        for a_k, a_v in dct.items():
            dct_simple[a_v] = a_k
    # Add grammar
    if len(dct_simple.keys()) > 0:
        for k, v in grammar.items():
            dct_simple[k] = v["message"]
    
    return dct_simple

def get_lines_to_test(row):
    pyright = row["lines_pyright"]
    mypy = row["lines_mypy"]
    s = ""
    if isinstance(pyright, str) and not isinstance(mypy, str):
        s = pyright
    elif not isinstance(pyright, str) and isinstance(mypy, str):
        s = mypy
    else:
        s = mypy
    return s

def get_execute_message(row):
    pyright = row["result_execute_pyright"]
    mypy = row["result_execute_mypy"]
    s = ""
    if isinstance(pyright, str) and not isinstance(mypy, str):
        s = pyright
    elif not isinstance(pyright, str) and isinstance(mypy, str):
        s = mypy
    else:
        s = mypy
    return s

def get_static_response(dct):
    s = ""
    for k,v in dct.items():
        if isinstance(v, dict):
            s += f'{k} : {v["message"]}\n'
        else:
            s += f'{k} : {v}\n'
    return s

def get_static_code(text):
    if text != "":
        return 0
    return 1

def get_error_type(text):
    # Regex to match and extract 'AssertionError'
    match = re.search(r"type:\s*<class '([\w.]+)'>", text)
    
    if match:
        extracted_error = match.group(1)
        #if extracted_error == "AssertionError":
            #return "passed"
        #else:
        return extracted_error
    else:
        return text

def get_id_number(file_completion):
    file_completion = file_completion.replace('.py','')
    idx = file_completion.find('_')
    file_number = file_completion[idx+1:]
    idx2 =  file_number.find('_')
    if idx2 > -1:
        return int(file_number[:idx2])
    else:
        return int(file_number)

def get_label_validation(result):
    if result == "pass":
        return 0
    return 1

def execute_validation(r):
    if "AssertionError" in r:
        return 1
    elif "failed: ," in r:
        return 1
    elif r == "failed: ":
        return 1
    elif "failed:" in r:
        return 0
    else:
        return 1
        
def get_df_per_program(df):
    grouped_df = df.sort_values(['validation'], ascending=[False])
    grouped_df = grouped_df[grouped_df['lines'].notna()]
    result = grouped_df.groupby('completion_id').head(1).reset_index(drop=True)
    return result

def odex_validator_results(validation_path, answers_path, benchmark="DS", split_lines=True, ifs_analysis=False, df_filter=df_filter):
    df_answers = pd.read_json(answers_path, lines=True)
    df_answers["execute"] = df_answers["result"].apply(execute_validation)
    df_answers["errors"] =  ~( df_answers["execute"] | df_answers["passed"].astype(bool) )
    df = pd.read_json(validation_path, lines=True)  # 'lines=True' for JSON Lines
    if ifs_analysis:
        df_answers["error_type"] = df_answers["result"].apply(get_error_type)
    #print(df_answers["error_type"])
    df["validation"] = df["passed"].apply(get_label_validation)
    # Get model name from validation file
    match1 = re.search(r"\/(.*?)_pyright_validation\.jsonl", validation_path)
    if match1:
        df["model"] =  match1.group(1).split('/')[-1]
        df_answers["model"] =  match1.group(1).split('/')[-1]
    else:
        match2 = re.search(r"\/(.*?)_pyright_validation\.jsonl", validation_path)
        if match2:
             df["model"] =  match2.group(1).split('/')[-1]
             df_answers["model"] =  match2.group(1).split('/')[-1]
    if benchmark == "DS":
        df["completion_id"] = df["completion_id"].apply(get_id_number)
        if split_lines:
            df = get_df_per_program(df)
        result = pd.merge(df, df_answers, on=['completion_id','model'], how='outer')
    else:
        df["id"] = df["completion_id"].apply(get_id_number)
        result = pd.merge(df, df_answers, on=['cid','id'], how='outer') 
    if ifs_analysis == 'syntax':
        result = result[result['error_type'].isin(['SyntaxError'])]
    elif ifs_analysis:
        result = result[~result['completion_id'].isin(df_filter['completion_id'])]
        result = result[result['error_type'].isin(['ImportError','ModuleNotFoundError','AttributeError','TypeError'])]
    return (result["errors"], result["validation"]), result

def odex_grammar_results(validation_path, answers_path, benchmark="DS", split_lines=True, ifs_analysis=False, df_filter=df_filter):
    df_answers = pd.read_json(answers_path, lines=True)
    df_answers["execute"] = df_answers["result"].apply(execute_validation)
    df_answers["errors"] =  ~( df_answers["execute"] | df_answers["passed"].astype(bool) )
    df = pd.read_json(validation_path, lines=True)  # 'lines=True' for JSON Lines
    df["validation"] = df["passed"].apply(get_label_validation)
    if ifs_analysis:
        df_answers["error_type"] = df_answers["result"].apply(get_error_type)
    # Get model name from validation file
    match1 = re.search(r"\/(.*?)_grammar_validation\.jsonl", validation_path)
    if match1:
        df["model"] =  match1.group(1).split('/')[-1]
        df_answers["model"] =  match1.group(1).split('/')[-1]
    else:
        match2 = re.search(r"\/(.*?)_grammar_validation\.jsonl", validation_path)
        if match2:
             df["model"] =  match2.group(1).split('/')[-1]
             df_answers["model"] =  match2.group(1).split('/')[-1]
    if benchmark == "DS":
        df["completion_id"] = df["completion_id"].apply(get_id_number)
        if split_lines:
            df = get_df_per_program(df)
        result = pd.merge(df, df_answers, on=['completion_id','model'], how='outer')
    else:
        df_answers["cid"] = df.index
        df["id"] = df["completion_id"].apply(get_id_number)
        result = pd.merge(df, df_answers, on=['cid','id'], how='outer')
    if ifs_analysis == 'syntax':
        result = result[result['error_type'].isin(['SyntaxError'])]
    elif ifs_analysis:
        result = result[result['error_type'].isin(['ImportError','ModuleNotFoundError','AttributeError','TypeError'])]
        result = result[~result['completion_id'].isin(df_filter['completion_id'])]
            
    return (result["errors"], result["validation"]), result

def flattened_analysis_pyright(json_output, key=["summary","errorCount"]):
    if isinstance(json_output, str):
        j = json.loads(json_output)
        dct = {}
        if "generalDiagnostics" in j:
            for err in j["generalDiagnostics"]:
                if "severity" in err:
                    if err["severity"] == "error":
                        k = f'line {err["range"]["start"]["line"]}, column {err["range"]["start"]["character"]}'
                        #k = f'line {err["range"]["start"]["line"]}'
                        dct[k] = {}
                        if "message" in err:
                            dct[k]["message"] = err["message"]
                        if "code" in err:
                            dct[k]["code"] = err["rule"]
        return dct
    if math.isnan(json_output):
        return json_output

def convert_mypy_dct(err, dct):
    if "severity" in err:
        if err["severity"] == "error":
            k = f'line {str(int(err["line"])-1)}, column {err["column"]}'
            # k = f'line {str(int(err["line"])-1)}'
            dct[k] = {}
            if "message" in err:
                dct[k]["message"] = err["message"]
            if "code" in err:
                dct[k]["code"] = err["code"]
    return dct

def flattened_analysis_mypy(json_output, key=["summary","errorCount"]):
    if isinstance(json_output, str):
        json_output = json_output.split("\n")
        dct = {}
        for f in json_output:
            if f != "":
                try:
                    j = json.loads(f)
                    dct = convert_mypy_dct(j, dct)
                except:
                    j = {
                        "file": f[0:f.find(".py")],
                        "line":0,
                        "column":0,
                        "message": f[f.find(":"):],
                        "hint": "",
                        "code": "",
                        "severity":"error"
                    }
                    dct = convert_mypy_dct(j, dct)
        return dct
    if math.isnan(json_output):
        return json_output

def get_line_with_word(g, w):
    for line in g:
        if w in line:
            return line
    return ""
        

def clean_analysis_grammar(grammar_output):
    dct = {}
    if "Input string is invalid" in grammar_output:
        g = grammar_output.split('\n')
        w = ""
        st = g[1] + '\n' + get_line_with_word(g, "\x1b") + '\n'
        j = {
            "line": 1,
            "column": 0,
            "message": st,
            "hint": "",
            "code": "",
            "severity": "error"
        }
        return convert_mypy_dct(j, dct)
    else:
        return dct
def flattened_analysis_pyright(json_output, key=["summary","errorCount"]):
    if isinstance(json_output, str):
        j = json.loads(json_output)
        dct = {}
        if "generalDiagnostics" in j:
            for err in j["generalDiagnostics"]:
                if "severity" in err:
                    if err["severity"] == "error":
                        k = f'line {err["range"]["start"]["line"]}, column {err["range"]["start"]["character"]}'
                        #k = f'line {err["range"]["start"]["line"]}'
                        dct[k] = {}
                        if "message" in err:
                            dct[k]["message"] = err["message"]
                        if "code" in err:
                            dct[k]["code"] = err["rule"]
        return dct
    if math.isnan(json_output):
        return json_output

def convert_mypy_dct(err, dct):
    if "severity" in err:
        if err["severity"] == "error":
            k = f'line {str(int(err["line"])-1)}, column {err["column"]}'
            # k = f'line {str(int(err["line"])-1)}'
            dct[k] = {}
            if "message" in err:
                dct[k]["message"] = err["message"]
            if "code" in err:
                dct[k]["code"] = err["code"]
    return dct

def flattened_analysis_mypy(json_output, key=["summary","errorCount"]):
    if isinstance(json_output, str):
        json_output = json_output.split("\n")
        dct = {}
        for f in json_output:
            if f != "":
                try:
                    j = json.loads(f)
                    dct = convert_mypy_dct(j, dct)
                except:
                    j = {
                        "file": f[0:f.find(".py")],
                        "line":0,
                        "column":0,
                        "message": f[f.find(":"):],
                        "hint": "",
                        "code": "",
                        "severity":"error"
                    }
                    dct = convert_mypy_dct(j, dct)
        return dct
    if math.isnan(json_output):
        return json_output

def get_line_with_word(g, w):
    for line in g:
        if w in line:
            return line
    return ""
        

def clean_analysis_grammar(grammar_output):
    dct = {}
    if "Input string is invalid" in grammar_output:
        g = grammar_output.split('\n')
        w = ""
        st = g[1] + '\n' + get_line_with_word(g, "\x1b") + '\n'
        j = {
            "line": 1,
            "column": 0,
            "message": st,
            "hint": "",
            "code": "",
            "severity": "error"
        }
        return convert_mypy_dct(j, dct)
    else:
        return dct
def static_check(df):
    if df["pyright"] == 0 and df["mypy"] == 0:
       return 0
    return 1
    

In [28]:
# DS-1000
file_names = ["results/errors/claude-3-opus-20240229-0-errors.jsonl", 
                "results/errors/gpt-3.5-turbo-0125-0-errors.jsonl", 
              "results/errors/gpt-4-0613-0-errors.jsonl",
              "results/errors/granite-code-latest-no-grammar-0-errors.jsonl"]

validation_files = ["results/dsb/mypy/claude-3-opus-20240229-0_pyright_validation.jsonl", 
                "results/dsb/mypy/gpt-3.5-0125-0_pyright_validation.jsonl", 
              "results/dsb/mypy/gpt-4-0613_pyright_validation.jsonl",
             "results/dsb/mypy/granite-code-latest-no-grammar-0_pyright_validation.jsonl"]

true = []
labels = []
pyright_df = pd.DataFrame()
for i in range(0, len(file_names)):
    #x, rs = odex_validator_results(validation_files[i], file_names[i], benchmark="DS", split_lines=False, ifs_analysis=True)
    #x, rs = odex_validator_results(validation_files[i], file_names[i], benchmark="DS", split_lines=False, ifs_analysis=False)
    x, rs = odex_validator_results(validation_files[i], file_names[i], benchmark="DS", split_lines=False, ifs_analysis=True)
    true = true + x[0].values.tolist()
    labels = labels + x[1].values.tolist()
    print(validation_files[i])
    x_1 = x[0].values.tolist()
    x_2 = x[1].values.tolist()
    print(confusion_matrix(x_1, x_2))
    print(classification_report(x_1, x_2))
    missing_labels = [] #[True for j in range(len(x_1), 1000)]
    true = true + missing_labels
    labels = labels + missing_labels
    pyright_df = pd.concat([pyright_df, rs])
#print(confusion_matrix(true, labels))
#print(classification_report(true, labels))

results/dsb/mypy/claude-3-opus-20240229-0_pyright_validation.jsonl
[[ 0  0]
 [54 26]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.33      0.49        80

    accuracy                           0.33        80
   macro avg       0.50      0.16      0.25        80
weighted avg       1.00      0.33      0.49        80

results/dsb/mypy/gpt-3.5-0125-0_pyright_validation.jsonl
[[ 0  0]
 [63 26]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.29      0.45        89

    accuracy                           0.29        89
   macro avg       0.50      0.15      0.23        89
weighted avg       1.00      0.29      0.45        89

results/dsb/mypy/gpt-4-0613_pyright_validation.jsonl
[[ 0  0]
 [52 17]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
   

/opt/homebrew/anaconda3/envs/interpret/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/anaconda3/envs/interpret/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/anaconda3/envs/interpret/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [8]:
# DS-1000
file_names = ["results/errors/claude-3-opus-20240229-0-errors.jsonl", 
                "results/errors/gpt-3.5-turbo-0125-0-errors.jsonl", 
              "results/errors/gpt-4-0613-0-errors.jsonl",
              "results/errors/granite-code-latest-no-grammar-0-errors.jsonl"]

validation_files = ["results/dsb/mypy/claude-3-opus-20240229-0_pyright_validation.jsonl", 
                "results/dsb/mypy/gpt-3.5-0125-0_pyright_validation.jsonl", 
              "results/dsb/mypy/gpt-4-0613_pyright_validation.jsonl",
             "results/dsb/mypy/granite-code-latest-no-grammar-0_pyright_validation.jsonl"]

true = []
labels = []
mypy_df = pd.DataFrame()
for i in range(0, len(file_names)):
    #x, rs = odex_validator_results(validation_files[i], file_names[i], benchmark="DS", split_lines=False, ifs_analysis=True)
    #x, rs = odex_validator_results(validation_files[i], file_names[i], benchmark="DS", split_lines=False, ifs_analysis=False)
    x, rs = odex_validator_results(validation_files[i], file_names[i], benchmark="DS", split_lines=False, ifs_analysis=True)
    true = true + x[0].values.tolist()
    labels = labels + x[1].values.tolist()
    print(validation_files[i])
    x_1 = x[0].values.tolist()
    x_2 = x[1].values.tolist()
    print(confusion_matrix(x_1, x_2))
    print(classification_report(x_1, x_2))
    missing_labels = []#[True for j in range(len(x_1), 1000)]
    true = true + missing_labels
    labels = labels + missing_labels
    mypy_df = pd.concat([mypy_df, rs])
    
#print(confusion_matrix(true, labels))
#print(classification_report(true, labels))

results/dsb/mypy/claude-3-opus-20240229-0_pyright_validation.jsonl
[[ 0  0]
 [54 26]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.33      0.49        80

    accuracy                           0.33        80
   macro avg       0.50      0.16      0.25        80
weighted avg       1.00      0.33      0.49        80

results/dsb/mypy/gpt-3.5-0125-0_pyright_validation.jsonl
[[ 0  0]
 [63 26]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.29      0.45        89

    accuracy                           0.29        89
   macro avg       0.50      0.15      0.23        89
weighted avg       1.00      0.29      0.45        89

results/dsb/mypy/gpt-4-0613_pyright_validation.jsonl
[[ 0  0]
 [52 17]]
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
   

/opt/homebrew/anaconda3/envs/interpret/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/anaconda3/envs/interpret/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/anaconda3/envs/interpret/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

In [22]:
# DS-1000
# Grammar
file_names = ["results/errors/claude-3-opus-20240229-0-errors.jsonl", 
                "results/errors/gpt-3.5-turbo-0125-0-errors.jsonl", 
              "results/errors/gpt-4-0613-0-errors.jsonl",
              "results/errors/granite-code-latest-no-grammar-0-errors.jsonl"]

validation_files = ["results/grammar/odex/claude-3-opus-20240229-0-odex_grammar_validation.jsonl", 
                "results/grammar/odex/gpt-3.5-turbo-0125-0-odex_grammar_validation.jsonl", 
              "results/grammar/odex/gpt-4-0613-0-odex_grammar_validation.jsonl",
             "results/grammar/odex/granite-code-latest-0_grammar_validation.jsonl"]

true = []
labels = []
grammar_df = pd.DataFrame()
for i in range(0, len(file_names)):
    #x, rs = odex_grammar_results(validation_files[i], file_names[i], benchmark="DS", split_lines=False, ifs_analysis=True)
    #x, rs = odex_grammar_results(validation_files[i], file_names[i], benchmark="DS", split_lines=False, ifs_analysis=False)
    x, rs = odex_grammar_results(validation_files[i], file_names[i], benchmark="DS", split_lines=False, ifs_analysis=True)
    df = pd.DataFrame({'true': x[0].values, 'pred': x[1].values}).dropna()
    x_1 = df['true'].tolist()
    x_2 = df['pred'].tolist()
    true = true + x_1
    labels = labels + x_2
    print(validation_files[i])
    #df = pd.DataFrame({'true': x[0].values, 'pred': x[1].values}).dropna()
    #x_1 = df['true'].tolist()
    #x_2 = df['pred'].tolist()
    #print(confusion_matrix(x_1, x_2))
    #print(classification_report(x_1, x_2))
    missing_labels = [] #[True for j in range(len(x_1), 1000)]
    true = true + missing_labels
    labels = labels + missing_labels
    grammar_df = pd.concat([grammar_df, rs])
print(confusion_matrix(true, labels))
print(classification_report(true, labels))

results/grammar/odex/claude-3-opus-20240229-0-odex_grammar_validation.jsonl
results/grammar/odex/gpt-3.5-turbo-0125-0-odex_grammar_validation.jsonl
results/grammar/odex/gpt-4-0613-0-odex_grammar_validation.jsonl
results/grammar/odex/granite-code-latest-0_grammar_validation.jsonl


ValueError: Found empty input array (e.g., `y_true` or `y_pred`) while a minimum of 1 sample is required.

In [30]:
mypy_df = mypy_df[['result_x', 'completion_id', 'model','lines', 'result_y']]
pyright_df = pyright_df[['result_x', 'completion_id', 'model','lines', 'result_y']]
grammar_df = grammar_df[['result_x', 'completion_id', 'model','lines', 'result_y']]
# rename result_x for result_static
# rename result_y for result_execute
mypy_df = mypy_df.rename(columns={'result_x':'result_static', 'result_y':'result_execute'})
pyright_df = pyright_df.rename(columns={'result_x':'result_static', 'result_y':'result_execute'})
grammar_df = grammar_df.rename(columns={'result_x':'result_static_grammar', 'result_y':'result_execute'})
detector_msg = pd.merge(pyright_df, mypy_df, on=['completion_id','model'], how='outer', suffixes=("_pyright", "_mypy"))
detector_msg = pd.merge(detector_msg, grammar_df, on=['completion_id','model'], how='outer', suffixes=("", "_grammar"))
detector_msg["result_static_pyright_unjson"] = detector_msg["result_static_pyright"].apply(flattened_analysis_pyright)
detector_msg["result_static_mypy_unjson"] = detector_msg["result_static_mypy"].apply(flattened_analysis_mypy)
detector_msg["result_static_grammar_unjson"] = detector_msg["result_static_grammar"].apply(clean_analysis_grammar)
detector_msg["static_union"] = detector_msg.apply(merge_static_analysis_union, axis=1)
detector_msg["static_intersection"] = detector_msg.apply(merge_static_analysis_intersection, axis=1)
detector_msg["static_union_str"] = detector_msg["static_union"].apply(get_static_response)
detector_msg["static_intersection_str"] = detector_msg["static_intersection"].apply(get_static_response)
detector_msg["lines"] = detector_msg.apply(get_lines_to_test, axis=1)
detector_msg["pyright_str"] = detector_msg["result_static_pyright_unjson"].apply(get_static_response)
detector_msg["mypy_str"] = detector_msg["result_static_mypy_unjson"].apply(get_static_response)
detector_msg["grammar_str"] = detector_msg["result_static_grammar_unjson"].apply(get_static_response)
detector_msg["execute_message"] = detector_msg.apply(get_execute_message, axis=1)
detector_msg["execute"] = detector_msg["execute_message"].apply(execute_validation)
detector_msg["error_type"] = detector_msg["execute_message"].apply(get_error_type)

In [36]:
new_ds_summary = detector_msg[['completion_id','model','pyright_str','mypy_str','grammar_str','execute_message','error_type']]
new_ds_summary.to_csv('ds1000_all_responses.csv', index=False)

In [37]:
new_ds_summary

,completion_id,model,pyright_str,mypy_str,grammar_str,execute_message,error_type
0,0,claude-3-opus-20240229-0,,"line 8, column 7 : ""EllipsisType"" not callable\n",,passed,passed
1,0,gpt-3.5-0125-0,,"line 8, column 7 : ""EllipsisType"" not callable\n",,"failed: 'result', type: <class 'KeyError'>",KeyError
2,0,gpt-4-0613,,"line 8, column 7 : ""EllipsisType"" not callable\n",,passed,passed
3,0,granite-code-latest-no-grammar-0,"line 9, column 0 : ""END"" is not defined\nline ...","line -1, column 0 : )\n",,"failed: 'result', type: <class 'KeyError'>",KeyError
4,1,claude-3-opus-20240229-0,,"line 8, column 7 : ""EllipsisType"" not callable\n",,failed: Can only compare identically-labeled S...,ValueError
...,...,...,...,...,...,...,...
3995,998,granite-code-latest-no-grammar-0,"line 3, column 4 : ""load_data"" is not defined\n","line 3, column 4 : Name ""load_data"" is not def...",,"failed: 'result', type: <class 'KeyError'>",KeyError
3996,999,claude-3-opus-20240229-0,,,"line 0, column 0 : Error: Unexpected character...","failed: batch2 must be a 3D tensor, type: <cla...",RuntimeError
3997,999,gpt-3.5-0125-0,,,"line 0, column 0 : Error: Unexpected character...",failed: Expected size for first two dimensions...,RuntimeError
3998,999,gpt-4-0613,,,,"failed: name 'hid_dim' is not defined, type: <...",NameError


In [41]:
ds = detector_msg[detector_msg['execute']==0]
ds_hallucinated = ds[ds['error_type'].isin(['ImportError','ModuleNotFoundError','AttributeError'])]
ds_hallucinated = ds_hallucinated[~ds_hallucinated['execute_message'].str.contains("No module named 'tensorflow'")]

In [42]:
type_errors_ds = ds[ds['error_type']=='TypeError']
#type_errors_message = type_errors_ds[type_errors_ds['execute_message'].str.contains('argument')]

In [39]:
np.unique(ds[ds['error_type']=='ModuleNotFoundError']['execute_message'])

array(["failed: No module named 'dask', type: <class 'ModuleNotFoundError'>",
       "failed: No module named 'numpy_indexed', type: <class 'ModuleNotFoundError'>",
       "failed: No module named 'pandas_datareader', type: <class 'ModuleNotFoundError'>",
       "failed: No module named 'tensorflow', type: <class 'ModuleNotFoundError'>",
       "failed: No module named 'xgboost', type: <class 'ModuleNotFoundError'>"],
      dtype=object)

In [46]:
new_ds = pd.concat([ds_hallucinated, type_errors_ds])

In [47]:
new_ds.groupby('model')['lines'].count()

model
claude-3-opus-20240229-0             44
gpt-3.5-0125-0                       50
gpt-4-0613                           34
granite-code-latest-no-grammar-0    114
Name: lines, dtype: int64

In [53]:
new_ds_summary = new_ds[['completion_id','model','pyright_str','mypy_str','grammar_str','execute_message','error_type']]

In [55]:
sampled_new_ds = new_ds_summary.groupby('model', group_keys=False).apply(lambda x: x if len(x) <= 50 else x.sample(n=50))

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_47970/3772664995.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_new_ds = new_ds_summary.groupby('model', group_keys=False).apply(lambda x: x if len(x) <= 50 else x.sample(n=50))


In [56]:
len(sampled_new_ds)

178

In [57]:
sampled_new_ds.to_csv('ds1000_messages.csv',index=False)

In [9]:
total = ds_detect_summary[ds_detect_summary['execute']==0].groupby(['model','execute','grammar'])['lines'].count().reset_index()
caught = total[total['grammar']==0]
all_errors = ds_detect_summary[ds_detect_summary['execute']==0].groupby(['model','execute'])['lines'].count().reset_index()
per_errors = pd.merge(caught,all_errors,on=['model','execute'])
per_errors['avg'] = per_errors['lines_x']/per_errors['lines_y']

In [10]:
per_errors

,model,execute,grammar,lines_x,lines_y,avg
0,claude-3-opus-20240229-0,0,0,35,255,0.137255
1,gpt-3.5-0125-0,0,0,29,256,0.113281
2,gpt-4-0613,0,0,32,219,0.146119
3,granite-code-latest-no-grammar-0,0,0,67,653,0.102603


In [7]:
ds_detect_summary = detector_msg[['completion_id','model','execute','error_type','execute_message','lines','pyright_str','mypy_str','grammar_str']]
ds_detect_summary['pyright'] = ds_detect_summary['pyright_str'].apply(lambda x : 0 if len(x) > 0 else 1)
ds_detect_summary['mypy'] = ds_detect_summary['mypy_str'].apply(lambda x : 0 if len(x) > 0 else 1)
ds_detect_summary['grammar'] = ds_detect_summary['grammar_str'].apply(lambda x : 0 if len(x) > 0 else 1)
ds_detect_summary['intersection'] = ds_detect_summary.apply(static_check, axis=1)

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_26152/1484906657.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds_detect_summary['pyright'] = ds_detect_summary['pyright_str'].apply(lambda x : 0 if len(x) > 0 else 1)
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_26152/1484906657.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ds_detect_summary['mypy'] = ds_detect_summary['mypy_str'].apply(lambda x : 0 if len(x) > 0 else 1)
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipy

In [9]:
ds_detect_summary.to_csv('ds_summary_explain.csv', index=False)

In [29]:
ds_detect_summary.groupby(['grammar','pyright','mypy','execute'])['lines'].count()

grammar  pyright  mypy  execute
0        0        0     0           134
                        1            87
                  1     0             2
                        1             1
         1        0     0             4
                        1            39
                  1     0            23
                        1            51
1        0        0     0           667
                        1           338
                  1     0            23
                        1            17
         1        0     0            86
                        1           442
                  1     0           444
                        1          1642
Name: lines, dtype: int64

In [25]:
801+90+25+18

934

In [30]:
134/163

0.8220858895705522

In [27]:
934/1383

0.6753434562545192

### False positive

In [35]:
fp = ds_detect_summary[ds_detect_summary["execute"]==1]
fp_static = fp[fp["intersection"]==0]
fp_grammar = fp[fp["grammar"]==0]

In [41]:
fp_static = fp[fp["intersection"]==0]

In [42]:
fp_grammar = fp[fp["grammar"]==0]

In [43]:
fp.groupby('model')['error_type'].count()

model
claude-3-opus-20240229-0            112
gpt-3.5-0125-0                      111
gpt-4-0613                          130
granite-code-latest-no-grammar-0     72
Name: error_type, dtype: int64

In [44]:
fp_grammar.groupby('model')['error_type'].count()

model
claude-3-opus-20240229-0            33
gpt-3.5-0125-0                      20
gpt-4-0613                          23
granite-code-latest-no-grammar-0    11
Name: error_type, dtype: int64

In [47]:
sampled_fp = fp.groupby('model', group_keys=False).apply(lambda x: x.sample(n=50))

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_48468/3262961898.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_fp = fp.groupby('model', group_keys=False).apply(lambda x: x.sample(n=50))


In [55]:
sampled_fp[['completion_id', 'model', 'execute', 'lines', 'pyright_str', 'mypy_str']].to_csv("static_fp.csv",index=False)

In [56]:
fp_grammar[['completion_id', 'model', 'execute', 'lines', 'grammar_str']].to_csv("grammar_fp.csv",index=False)

### False negative

In [57]:
fn = ds_detect_summary[ds_detect_summary["execute"]==0]
fn_static = fn[fn["intersection"]==1]
sampled_fn_static = fn_static.groupby('model', group_keys=False).apply(lambda x: x.sample(n=50))

,completion_id,model,execute,lines,pyright_str,mypy_str,grammar_str,error_type,execute_message,pyright,mypy,grammar,intersection
1,0,gpt-3.5-0125-0,0,import pandas as pd\nimport numpy as np\n\n\nd...,,"line 8, column 7 : ""EllipsisType"" not callable\n",,KeyError,"failed: 'result', type: <class 'KeyError'>",1,0,1,1
4,1,claude-3-opus-20240229-0,0,import pandas as pd\nimport numpy as np\n\n\nd...,,"line 8, column 7 : ""EllipsisType"" not callable\n",,ValueError,failed: Can only compare identically-labeled S...,1,0,1,1
5,1,gpt-3.5-0125-0,0,import pandas as pd\nimport numpy as np\n\n\nd...,,"line 8, column 7 : ""EllipsisType"" not callable\n",,KeyError,"failed: 'result', type: <class 'KeyError'>",1,0,1,1
6,1,gpt-4-0613,0,import pandas as pd\nimport numpy as np\n\n\nd...,,"line 8, column 7 : ""EllipsisType"" not callable\n",,ValueError,failed: Can only compare identically-labeled S...,1,0,1,1
7,1,granite-code-latest-no-grammar-0,0,import pandas as pd\nimport numpy as np\n\n\nd...,,"line 8, column 7 : ""EllipsisType"" not callable\n",,KeyError,"failed: 'result', type: <class 'KeyError'>",1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3852,963,claude-3-opus-20240229-0,0,import numpy as np\nimport pandas as pd\nimpor...,,,"line 0, column 0 : Error: Unexpected character...",RuntimeError,failed: The size of tensor a (1000) must match...,1,1,0,1
3996,999,claude-3-opus-20240229-0,0,import numpy as np\nimport pandas as pd\nimpor...,,,"line 0, column 0 : Error: Unexpected character...",RuntimeError,"failed: batch2 must be a 3D tensor, type: <cla...",1,1,0,1
3997,999,gpt-3.5-0125-0,0,import numpy as np\nimport pandas as pd\nimpor...,,,"line 0, column 0 : Error: Unexpected character...",RuntimeError,failed: Expected size for first two dimensions...,1,1,0,1
3998,999,gpt-4-0613,0,import numpy as np\nimport pandas as pd\nimpor...,,,,NameError,"failed: name 'hid_dim' is not defined, type: <...",1,1,1,1


In [59]:
fn_grammar = fn[fn["grammar"]==1]
fn_grammar

,completion_id,model,execute,lines,pyright_str,mypy_str,grammar_str,error_type,execute_message,pyright,mypy,grammar,intersection
1,0,gpt-3.5-0125-0,0,import pandas as pd\nimport numpy as np\n\n\nd...,,"line 8, column 7 : ""EllipsisType"" not callable\n",,KeyError,"failed: 'result', type: <class 'KeyError'>",1,0,1,1
3,0,granite-code-latest-no-grammar-0,0,import pandas as pd\nimport numpy as np\n\n\nd...,"line 9, column 0 : ""END"" is not defined\nline ...","line -1, column 0 : )\n",,KeyError,"failed: 'result', type: <class 'KeyError'>",0,0,1,0
4,1,claude-3-opus-20240229-0,0,import pandas as pd\nimport numpy as np\n\n\nd...,,"line 8, column 7 : ""EllipsisType"" not callable\n",,ValueError,failed: Can only compare identically-labeled S...,1,0,1,1
5,1,gpt-3.5-0125-0,0,import pandas as pd\nimport numpy as np\n\n\nd...,,"line 8, column 7 : ""EllipsisType"" not callable\n",,KeyError,"failed: 'result', type: <class 'KeyError'>",1,0,1,1
6,1,gpt-4-0613,0,import pandas as pd\nimport numpy as np\n\n\nd...,,"line 8, column 7 : ""EllipsisType"" not callable\n",,ValueError,failed: Can only compare identically-labeled S...,1,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3991,997,granite-code-latest-no-grammar-0,0,import numpy as np\nimport pandas as pd\nimpor...,"line 3, column 4 : ""load_data"" is not defined\...","line -1, column 0 : )\n",,KeyError,"failed: 'result', type: <class 'KeyError'>",0,0,1,0
3994,998,gpt-4-0613,0,import numpy as np\nimport pandas as pd\nimpor...,"line 3, column 4 : ""load_data"" is not defined\n","line 3, column 4 : Name ""load_data"" is not def...",,RuntimeError,failed: Sizes of tensors must match except in ...,0,0,1,0
3995,998,granite-code-latest-no-grammar-0,0,import numpy as np\nimport pandas as pd\nimpor...,"line 3, column 4 : ""load_data"" is not defined\n","line 3, column 4 : Name ""load_data"" is not def...",,KeyError,"failed: 'result', type: <class 'KeyError'>",0,0,1,0
3998,999,gpt-4-0613,0,import numpy as np\nimport pandas as pd\nimpor...,,,,NameError,"failed: name 'hid_dim' is not defined, type: <...",1,1,1,1


In [58]:
fn_static.groupby('model')['error_type'].count()

model
claude-3-opus-20240229-0             82
gpt-3.5-0125-0                      104
gpt-4-0613                           88
granite-code-latest-no-grammar-0    308
Name: error_type, dtype: int64

In [60]:
fn_grammar.groupby('model')['error_type'].count()

model
claude-3-opus-20240229-0            220
gpt-3.5-0125-0                      227
gpt-4-0613                          187
granite-code-latest-no-grammar-0    586
Name: error_type, dtype: int64

In [61]:
sampled_fn_static = fn_static.groupby('model', group_keys=False).apply(lambda x: x.sample(n=50))
sampled_fn_grammar = fn_grammar.groupby('model', group_keys=False).apply(lambda x: x.sample(n=50))

/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_48468/2031471264.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_fn_static = fn_static.groupby('model', group_keys=False).apply(lambda x: x.sample(n=50))
/var/folders/d8/_gbst69d4971d282tmc920nm0000gq/T/ipykernel_48468/2031471264.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_fn_grammar = fn_grammar.groupby('model', group_ke

In [66]:
sampled_fn_static[['completion_id', 'model', 'execute', 'lines', 'pyright_str', 'mypy_str', 'error_type', 'execute_message']].to_csv("static_fn.csv",index=False)

In [67]:
sampled_fn_grammar[['completion_id', 'model', 'execute', 'lines', 'grammar_str', 'error_type', 'execute_message']].to_csv("grammar_fn.csv",index=False)

### Overlap DS-1000
#### Caught

In [34]:
ds_v = ds_detect_summary[ds_detect_summary['execute']==0]
ds_v = ds_v[ds_v['error_type']!='passed']
ds_v = ds_v[ds_v['intersection']==0]

In [35]:
ds_v = ds_v[['model','completion_id','error_type', 'execute_message', 'lines','pyright_str', 'mypy_str', 'grammar_str']]

In [36]:
ds_v

,model,completion_id,error_type,execute_message,lines,pyright_str,mypy_str,grammar_str
3,granite-code-latest-no-grammar-0,0,KeyError,"failed: 'result', type: <class 'KeyError'>",import pandas as pd\nimport numpy as np\n\n\nd...,"line 9, column 0 : ""END"" is not defined\nline ...","line -1, column 0 : )\n",
16,claude-3-opus-20240229-0,4,IndentationError,failed: expected an indented block after funct...,import pandas as pd\n\nexample_df = pd.DataFra...,"line 10, column 0 : Expected indented block\n","line -1, column 0 : )\n",
19,granite-code-latest-no-grammar-0,4,IndentationError,failed: expected an indented block after funct...,import pandas as pd\n\nexample_df = pd.DataFra...,"line 10, column 0 : Expected indented block\n","line -1, column 0 : )\n",
31,granite-code-latest-no-grammar-0,7,NameError,"failed: name 'Dataframe' is not defined, type:...",import pandas as pd\n\n\ndf = pd.DataFrame({'u...,"line 5, column 10 : ""Dataframe"" is not defined\n","line 5, column 10 : Cannot determine type of ""...",
47,granite-code-latest-no-grammar-0,11,NameError,"failed: name 'tz_localize' is not defined, typ...",import pandas as pd\n\n\ndf = pd.DataFrame({'d...,"line 5, column 0 : ""tz_localize"" is not defined\n","line 5, column 0 : Name ""tz_localize"" is not d...",
...,...,...,...,...,...,...,...,...
3989,gpt-3.5-0125-0,997,KeyError,"failed: 'result', type: <class 'KeyError'>",import numpy as np\nimport pandas as pd\nimpor...,"line 3, column 4 : ""load_data"" is not defined\...","line 3, column 4 : Name ""load_data"" is not def...","line 0, column 0 : Error: Unexpected character..."
3991,granite-code-latest-no-grammar-0,997,KeyError,"failed: 'result', type: <class 'KeyError'>",import numpy as np\nimport pandas as pd\nimpor...,"line 3, column 4 : ""load_data"" is not defined\...","line -1, column 0 : )\n",
3993,gpt-3.5-0125-0,998,RuntimeError,failed: Sizes of tensors must match except in ...,import numpy as np\nimport pandas as pd\nimpor...,"line 3, column 4 : ""load_data"" is not defined\n","line 3, column 4 : Name ""load_data"" is not def...","line 0, column 0 : Error: Unexpected character..."
3994,gpt-4-0613,998,RuntimeError,failed: Sizes of tensors must match except in ...,import numpy as np\nimport pandas as pd\nimpor...,"line 3, column 4 : ""load_data"" is not defined\n","line 3, column 4 : Name ""load_data"" is not def...",


In [37]:
ds_v.to_csv('ds_detected_reasons.csv',index=False)

In [33]:
ds_g = ds_v[ds_v['grammar']==0]

### Percentage of detected by reason

In [3]:
ds = pd.read_csv('ds_detected_reasons.csv')

In [ ]:
df['grammar_str'].notna().sum()

In [4]:
ds

,model,completion_id,error_type,execute_message,lines,pyright_str,mypy_str,grammar_str,static_capable,grammar_capable
0,granite-code-latest-no-grammar-0,0,KeyError,"failed: 'result', type: <class 'KeyError'>",import pandas as pd\nimport numpy as np\n\n\nd...,"line 9, column 0 : ""END"" is not defined\nline ...","line -1, column 0 : )\n",NaN,NaN,NaN
1,claude-3-opus-20240229-0,4,IndentationError,failed: expected an indented block after funct...,import pandas as pd\n\nexample_df = pd.DataFra...,"line 10, column 0 : Expected indented block\n","line -1, column 0 : )\n",NaN,NaN,NaN
2,granite-code-latest-no-grammar-0,4,IndentationError,failed: expected an indented block after funct...,import pandas as pd\n\nexample_df = pd.DataFra...,"line 10, column 0 : Expected indented block\n","line -1, column 0 : )\n",NaN,NaN,NaN
3,granite-code-latest-no-grammar-0,7,NameError,"failed: name 'Dataframe' is not defined, type:...",import pandas as pd\n\n\ndf = pd.DataFrame({'u...,"line 5, column 10 : ""Dataframe"" is not defined\n","line 5, column 10 : Cannot determine type of ""...",NaN,NaN,NaN
4,granite-code-latest-no-grammar-0,11,NameError,"failed: name 'tz_localize' is not defined, typ...",import pandas as pd\n\n\ndf = pd.DataFrame({'d...,"line 5, column 0 : ""tz_localize"" is not defined\n","line 5, column 0 : Name ""tz_localize"" is not d...",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
657,gpt-3.5-0125-0,997,KeyError,"failed: 'result', type: <class 'KeyError'>",import numpy as np\nimport pandas as pd\nimpor...,"line 3, column 4 : ""load_data"" is not defined\...","line 3, column 4 : Name ""load_data"" is not def...","line 0, column 0 : Error: Unexpected character...",No,No
658,granite-code-latest-no-grammar-0,997,KeyError,"failed: 'result', type: <class 'KeyError'>",import numpy as np\nimport pandas as pd\nimpor...,"line 3, column 4 : ""load_data"" is not defined\...","line -1, column 0 : )\n",NaN,NaN,NaN
659,gpt-3.5-0125-0,998,RuntimeError,failed: Sizes of tensors must match except in ...,import numpy as np\nimport pandas as pd\nimpor...,"line 3, column 4 : ""load_data"" is not defined\n","line 3, column 4 : Name ""load_data"" is not def...","line 0, column 0 : Error: Unexpected character...",No,No
660,gpt-4-0613,998,RuntimeError,failed: Sizes of tensors must match except in ...,import numpy as np\nimport pandas as pd\nimpor...,"line 3, column 4 : ""load_data"" is not defined\n","line 3, column 4 : Name ""load_data"" is not def...",NaN,NaN,NaN


In [41]:
ds_g_none.to_csv('diff_ds.csv',index=False)

In [35]:
ds_s = ds_v[ds_v['intersection']==0]

In [25]:
ds_s

,model,completion_id,grammar,intersection,lines,execute
3,granite-code-latest-no-grammar-0,0,1,0,import pandas as pd\nimport numpy as np\n\n\nd...,0
16,claude-3-opus-20240229-0,4,1,0,import pandas as pd\n\nexample_df = pd.DataFra...,0
19,granite-code-latest-no-grammar-0,4,1,0,import pandas as pd\n\nexample_df = pd.DataFra...,0
31,granite-code-latest-no-grammar-0,7,1,0,import pandas as pd\n\n\ndf = pd.DataFrame({'u...,0
47,granite-code-latest-no-grammar-0,11,1,0,import pandas as pd\n\n\ndf = pd.DataFrame({'d...,0
...,...,...,...,...,...,...
3989,gpt-3.5-0125-0,997,0,0,import numpy as np\nimport pandas as pd\nimpor...,0
3991,granite-code-latest-no-grammar-0,997,1,0,import numpy as np\nimport pandas as pd\nimpor...,0
3993,gpt-3.5-0125-0,998,0,0,import numpy as np\nimport pandas as pd\nimpor...,0
3994,gpt-4-0613,998,1,0,import numpy as np\nimport pandas as pd\nimpor...,0


In [26]:
intersection = pd.merge(ds_g[['model','completion_id']], ds_s[['model','completion_id']], how='inner', on=['model','completion_id'])
intersection

,model,completion_id
0,granite-code-latest-no-grammar-0,12
1,granite-code-latest-no-grammar-0,118
2,granite-code-latest-no-grammar-0,121
3,claude-3-opus-20240229-0,194
4,gpt-3.5-0125-0,194
...,...,...
129,claude-3-opus-20240229-0,996
130,gpt-3.5-0125-0,996
131,granite-code-latest-no-grammar-0,996
132,gpt-3.5-0125-0,997


In [37]:
30/90

0.3333333333333333

In [39]:
29/30

0.9666666666666667

In [40]:
12.5-28.4

-15.899999999999999

In [41]:
10.5-23.7

-13.2

In [42]:
67.5+5.3

72.8

In [43]:
11.8+15.9

27.700000000000003

In [44]:
52.6 + 7.6

60.2

In [45]:
10.9 + 13.2

24.1